# AGRI-INTELLIGENCE | PLATFORM NOTEBOOK V1
**ArqTech Labs · Multimodal Agricultural Intelligence**

Este notebook é a interface de orquestração no Google Colab. A lógica reutilizável permanece em `src/agri_intelligence`; dados, estado e artefatos persistem no Google Drive.

**Milestone atual:** bootstrap persistente e recuperação verificável das fontes da Phase 2.5. Nenhum treinamento ou inferência é executado nesta versão inicial.

Branch de desenvolvimento: `feat/platform-notebook-v1`.

## 0. Regras de execução

- Não instalar dependências automaticamente.
- Não baixar pesos nem iniciar treinamento/inferência nesta etapa.
- Não carregar ortomosaicos completos na RAM; usar leitura por janelas.
- Recuperar arquivos somente quando ausentes ou com checksum divergente.
- Preservar arquivos inválidos como backup antes de substituí-los.
- Interromper em checksum divergente, fonte inesperada ou branch incorreta.
- Cada etapa registra configuração, commit, artefatos e resultado.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, platform

PROJECT_NAME = "Agri-Intelligence"
REPOSITORY = "edu-moraess/agri-intelligence"
REPOSITORY_URL = f"https://github.com/{REPOSITORY}.git"
DEVELOPMENT_BRANCH = "feat/platform-notebook-v1"
ZENODO_RECORD_ID = "15235602"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

DRIVE_ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
REPO_ROOT = Path("/content/agri-intelligence-platform-v1")
SOURCE_ROOT = DRIVE_ROOT / "data/raw/zenodo_15235602"
INTERIM_ROOT = DRIVE_ROOT / "data/interim/phase_2_5"
PROCESSED_ROOT = DRIVE_ROOT / "data/processed/phase_2_5"
STATE_ROOT = DRIVE_ROOT / "state"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
LOG_ROOT = DRIVE_ROOT / "logs"

EXPECTED_RASTER_MD5 = {
    "orthomosaic_1.tif": "062a5c9028fff72f3fc276917e0a5af1",
    "orthomosaic_2.tif": "c5c478aa73a22f11d52eb8176c4c8b1a",
    "orthomosaic_3.tif": "a800e5be8ec95f4a61d2c24a0b4e4a74",
}
EXPECTED_RASTER_SHA256 = {
    "orthomosaic_1.tif": "d4317d8eb402a15d33e084e131de26bd6340a14756148b4e6011a912f3695a2c",
    "orthomosaic_2.tif": "c5a81cfab297866a82633130f161ac3a31524956a17c9e94e98f8e6ac3583feb",
    "orthomosaic_3.tif": "86abb6bb4dca079aa07de0520e376e34fcaf21efaba93cfd587c520a4008948d",
}
EXPECTED_GT_COUNTS = {"gt_1": 61251, "gt_2": 30547, "gt_3": 55374}

print(f"{PROJECT_NAME} | PLATFORM V1")
print("Branch alvo:", DEVELOPMENT_BRANCH)
print("Drive:", DRIVE_ROOT)
print("Fonte:", ZENODO_API)

## 1. Montagem do armazenamento persistente

Os arquivos brutos e resultados ficam no Drive, não no disco efêmero de `/content`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

for path in [DRIVE_ROOT, SOURCE_ROOT, INTERIM_ROOT, PROCESSED_ROOT,
             STATE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / "manifests", LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PERSISTENT STORAGE: READY")
print("Source root:", SOURCE_ROOT)
print("State root:", STATE_ROOT)

## 2. Repositório e identidade da execução

Em uma sessão nova, o notebook clona a branch de desenvolvimento. Se o repositório já existir, não troca de branch nem sobrescreve alterações locais.

In [ ]:
def run_cmd(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if not (REPO_ROOT / ".git").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"Diretório existe e não é um clone Git limpo: {REPO_ROOT}")
    run_cmd(["git", "clone", "--branch", DEVELOPMENT_BRANCH,
             "--single-branch", REPOSITORY_URL, str(REPO_ROOT)])

current_branch = run_cmd(["git", "branch", "--show-current"], cwd=REPO_ROOT)
current_commit = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT)
working_tree = run_cmd(["git", "status", "--porcelain"], cwd=REPO_ROOT)

if current_branch != DEVELOPMENT_BRANCH:
    raise RuntimeError(
        f"Branch incorreta: {current_branch}. Esperada: {DEVELOPMENT_BRANCH}. "
        "Não foi feito checkout automático."
    )

sys.path.insert(0, str(REPO_ROOT / "src"))
print("BRANCH:", current_branch)
print("HEAD:", current_commit)
print("WORKTREE:", "CLEAN" if not working_tree else "MODIFIED")
if working_tree:
    print(working_tree)

### 2.1. Recuperar o checkpoint local da Phase 2.5

O bundle criado anteriormente no Drive pode conter o protocolo e os módulos L2 que não foram publicados no GitHub. Esta célula verifica e importa a referência para o clone novo **sem trocar de branch, fazer merge ou alterar a árvore de trabalho**.

In [ ]:
BUNDLE_PATH = Path("/content/drive/MyDrive/agri_phase25/phase25_protocol.bundle")
BUNDLE_REF = "refs/heads/feat/phase-2-5-first-real-experiment"

if not BUNDLE_PATH.is_file():
    print("PHASE 2.5 BUNDLE: NOT FOUND")
    print("A continuidade poderá ser reconstruída a partir da base remota, sem presumir que L2 esteja presente.")
else:
    verification = subprocess.run(
        ["git", "bundle", "verify", str(BUNDLE_PATH)],
        cwd=REPO_ROOT, text=True, capture_output=True
    )
    if verification.returncode != 0:
        raise RuntimeError("Bundle Phase 2.5 inválido: " + verification.stderr.strip())

    refs_output = run_cmd(["git", "bundle", "list-heads", str(BUNDLE_PATH)], cwd=REPO_ROOT)
    bundle_refs = [line.split()[-1] for line in refs_output.splitlines() if line.strip()]
    if BUNDLE_REF not in bundle_refs:
        raise RuntimeError(f"Referência esperada não encontrada no bundle. Refs: {bundle_refs}")

    run_cmd([
        "git", "fetch", str(BUNDLE_PATH),
        f"{BUNDLE_REF}:refs/remotes/backup/phase25"
    ], cwd=REPO_ROOT)
    bundle_commit = run_cmd(
        ["git", "rev-parse", "refs/remotes/backup/phase25"], cwd=REPO_ROOT
    )
    print("PHASE 2.5 BUNDLE: VERIFIED")
    print("Imported ref: refs/remotes/backup/phase25")
    print("Bundle commit:", bundle_commit)
    print("Current branch unchanged:", current_branch)
    bundle_state = {
        "bundle_path": str(BUNDLE_PATH),
        "source_ref": BUNDLE_REF,
        "imported_ref": "refs/remotes/backup/phase25",
        "commit": bundle_commit,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    bundle_state_path = STATE_ROOT / "phase_2_5_bundle_recovery.json"
    bundle_state_temp = bundle_state_path.with_name(bundle_state_path.name + ".part")
    bundle_state_temp.write_text(json.dumps(bundle_state, indent=2), encoding="utf-8")
    bundle_state_temp.replace(bundle_state_path)

## 3. Compatibilidade do ambiente

Esta etapa apenas detecta dependências. Não instala nem atualiza pacotes.

In [ ]:
import importlib.util
import sys

OPTIONAL_MODULES = ["numpy", "pandas", "rasterio", "geopandas", "shapely", "requests", "PIL"]
availability = {name: importlib.util.find_spec(name) is not None for name in OPTIONAL_MODULES}

runtime = {
    "python": sys.version,
    "platform": platform.platform(),
    "modules": availability,
    "branch": current_branch,
    "commit": current_commit,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}

STATE_ROOT.mkdir(parents=True, exist_ok=True)
(STATE_ROOT / "runtime_environment.json").write_text(
    json.dumps(runtime, indent=2), encoding="utf-8"
)

for name, available in availability.items():
    print(f"{'AVAILABLE' if available else 'MISSING'} | {name}")
print("Python:", sys.version.split()[0])
print("Nenhuma instalação automática foi executada.")

## 4. Recuperação persistente das fontes — Zenodo 15235602

O conjunto contém três GeoTIFFs e três famílias Shapefile. Cada Shapefile depende de arquivos auxiliares (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`; e metadados `.qmd` quando publicados). O processo abaixo seleciona somente os ortomosaicos e GT. Não utiliza `HC_bboxes_*` nem `tiles_*` como ground truth.

A recuperação é retomável: arquivos válidos são preservados; downloads são gravados como `.part`, verificados por tamanho e MD5 do registro e só então renomeados. SHA-256 é calculado para a proveniência local.

In [ ]:
import requests, shutil, time

def digest_file(path, algorithm="sha256", chunk_size=8 * 1024 * 1024):
    h = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".part")
    temp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

response = requests.get(ZENODO_API, timeout=(30, 120))
response.raise_for_status()
record = response.json()
if str(record.get("id")) != ZENODO_RECORD_ID:
    raise RuntimeError("O registro Zenodo retornado não corresponde ao ID solicitado.")

raw_files = record.get("files", [])
if isinstance(raw_files, dict):
    if isinstance(raw_files.get("entries"), list):
        raw_files = raw_files["entries"]
    else:
        raw_files = list(raw_files.values())
file_index = {item.get("key") or item.get("filename"): item for item in raw_files}

core_names = [f"orthomosaic_{i}.tif" for i in (1, 2, 3)]
gt_extensions = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
gt_names = [f"gt_{i}{ext}" for i in (1, 2, 3) for ext in gt_extensions]
optional_names = [f"gt_{i}.qmd" for i in (1, 2, 3)]
required_names = core_names + gt_names
missing_metadata = sorted(set(required_names) - set(file_index))
if missing_metadata:
    raise RuntimeError(f"Arquivos obrigatórios ausentes no registro: {missing_metadata}")

metadata = record.get("metadata", {})
license_metadata = metadata.get("license") or metadata.get("rights")
print("Registro:", record.get("doi") or f"Zenodo {ZENODO_RECORD_ID}")
print("Título:", metadata.get("title", "não informado"))
print("Licença declarada na API:", license_metadata)
if not license_metadata:
    print("ATENÇÃO: API não expôs licença estruturada; manter ressalva documental.")

selected_names = required_names + [n for n in optional_names if n in file_index]
recovery_inventory = []

for index, name in enumerate(selected_names, start=1):
    info = file_index[name]
    expected_size = int(info.get("size", -1))
    checksum = info.get("checksum", "")
    if not checksum.startswith("md5:") or expected_size < 0:
        raise RuntimeError(f"Metadados de tamanho/checksum inválidos para {name}: {info}")
    expected_md5 = checksum.split(":", 1)[1].lower()
    target = SOURCE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)

    valid_existing = (
        target.is_file()
        and target.stat().st_size == expected_size
        and digest_file(target, "md5") == expected_md5
    )
    if valid_existing:
        print(f"[{index}/{len(selected_names)}] VALIDADO | {name}")
    else:
        if target.exists():
            stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            backup = target.with_name(target.name + f".invalid-{stamp}")
            target.replace(backup)
            print(f"BACKUP DE ARQUIVO INVÁLIDO | {backup.name}")

        links = info.get("links", {})
        download_url = links.get("download") or links.get("content") or links.get("self")
        if not download_url:
            raise RuntimeError(f"URL de download ausente para {name}")

        temp = target.with_name(target.name + ".part")
        md5 = hashlib.md5()
        sha256 = hashlib.sha256()
        size = 0
        print(f"[{index}/{len(selected_names)}] BAIXANDO | {name} | {expected_size / 1e9:.3f} GB")
        with requests.get(download_url, stream=True, timeout=(30, 240)) as download:
            download.raise_for_status()
            with open(temp, "wb") as output:
                for chunk in download.iter_content(chunk_size=8 * 1024 * 1024):
                    if not chunk:
                        continue
                    output.write(chunk)
                    md5.update(chunk)
                    sha256.update(chunk)
                    size += len(chunk)
        if size != expected_size or md5.hexdigest().lower() != expected_md5:
            temp.unlink(missing_ok=True)
            raise RuntimeError(
                f"Integridade divergente em {name}: size={size}/{expected_size}, "
                f"md5={md5.hexdigest()}/{expected_md5}"
            )
        temp.replace(target)

    local_sha256 = digest_file(target, "sha256")
    if name in EXPECTED_RASTER_SHA256 and local_sha256 != EXPECTED_RASTER_SHA256[name]:
        raise RuntimeError(f"SHA-256 histórico divergente para {name}: {local_sha256}")
    recovery_inventory.append({
        "name": name,
        "relative_path": name,
        "size_bytes": target.stat().st_size,
        "zenodo_md5": expected_md5,
        "local_sha256": local_sha256,
        "download_url": download_url if not valid_existing else None,
        "status": "VERIFIED",
    })

source_manifest = {
    "dataset_id": "maize_seedling_orthomosaics_zenodo_15235602",
    "record_id": ZENODO_RECORD_ID,
    "doi": record.get("doi", "10.5281/zenodo.15235602"),
    "record_version": metadata.get("version"),
    "title": metadata.get("title"),
    "license_metadata": license_metadata,
    "license_caveat": "Preservar a ressalva: direitos de terceiros do depositante não foram verificados independentemente.",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_root": str(SOURCE_ROOT),
    "files": recovery_inventory,
    "result": "PASS",
}
atomic_json(ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json", source_manifest)
print("\nSOURCE RECOVERY: PASS")
print("Arquivos recuperados/validados:", len(recovery_inventory))
print("Dados persistidos em:", SOURCE_ROOT)
print("Manifesto:", ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json")

## 5. Caracterização do ambiente e fontes

Com os arquivos persistidos e verificados, a próxima etapa será reabrir os GeoTIFFs por janela, confirmar dimensões/CRS/transform/bandas e ler os GT Shapefiles. Esses dados serão ligados ao manifesto existente. A implementação espacial e a materialização do dataset ficam em módulos testáveis no repositório, não neste notebook.

## 6. Roadmap operacional

| Marco | Objetivo | Estado |
|---|---|---|
| Bootstrap | Drive, branch, ambiente e identidade de execução | Implementado neste notebook |
| Recovery | Recuperar fontes e validar checksums | Implementado neste notebook |
| L3 | Seleção espacial, deduplicação e splits | Próximo |
| L4 | Materialização RGB/YOLO e conservação de GT | Pendente |
| L5 | Manifesto, integridade e leakage espacial | Pendente |
| L6 | Smoke test e treinamento controlado | Bloqueado pelos gates de dados |
| L7 | Avaliação e relatório reproduzível | Pendente |

Depois da primeira prova de percepção, a plataforma evolui para Sensor Intelligence, Environmental Intelligence, Multimodal Fusion, Temporal Intelligence, Agricultural Context, Predictive/Anomaly Intelligence e Decision Intelligence.

## 7. Limites deste milestone

- Este notebook ainda não declara L3, L4 ou L5 aprovadas.
- Nenhum treinamento ou inferência foi executado.
- A recuperação do dataset não equivale à aprovação jurídica de todos os direitos de uso.
- Os resultados agronômicos exigem validação independente em campo.
- O próximo incremento deverá reutilizar os contratos e módulos já existentes, sem duplicar Data Engine, percepção ou avaliação.

In [3]:
from pathlib import Path
import subprocess

CANDIDATES = [
    Path("/content/agri-intelligence-platform-v1"),
    Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/agri-intelligence-platform-v1"),
    Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence"),
]

def find_repo():
    for candidate in CANDIDATES:
        if (candidate / ".git").exists():
            return candidate

    content = Path("/content")
    for git_dir in content.glob("*/.git"):
        if git_dir.is_dir() or git_dir.is_file():
            if git_dir.parent.name == "agri-intelligence-platform-v1":
                return git_dir.parent

    return None

repo = find_repo()

print("=" * 65)
print("AGRI-INTELLIGENCE — LOCALIZAÇÃO DO REPOSITÓRIO")
print("=" * 65)

if repo is None:
    print("STATUS: REPOSITÓRIO NÃO LOCALIZADO")
    print("\nPastas disponíveis em /content:")
    for p in Path("/content").iterdir():
        print(" -", p)
else:
    print("REPOSITÓRIO:", repo)

    def git(args):
        return subprocess.run(
            ["git", *args],
            cwd=repo,
            capture_output=True,
            text=True,
            check=True
        )

    print("BRANCH:", git(["branch", "--show-current"]).stdout.strip())
    print("HEAD:", git(["rev-parse", "HEAD"]).stdout.strip())
    print("WORKTREE:")
    print(git(["status", "--short"]).stdout or "CLEAN")

    print("\nCHECKPOINT:")
    print(
        git([
            "log", "-1",
            "--format=%H%n%s"
        ]).stdout.strip()
    )

print("=" * 65)

AGRI-INTELLIGENCE — LOCALIZAÇÃO DO REPOSITÓRIO
STATUS: REPOSITÓRIO NÃO LOCALIZADO

Pastas disponíveis em /content:
 - /content/.config
 - /content/sample_data


In [4]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

print("=" * 65)
print("AGRI-INTELLIGENCE — RECUPERAÇÃO DO CHECKPOINT")
print("=" * 65)

if not ROOT.exists():
    print("ERRO: pasta principal não encontrada:", ROOT)
else:
    print("DRIVE ROOT:", ROOT)
    print("\nCONTEÚDO PRINCIPAL:")

    for p in sorted(ROOT.iterdir()):
        tipo = "DIR " if p.is_dir() else "FILE"
        print(f"[{tipo}] {p.name}")

    backup = ROOT / "artifacts" / "backups" / "notebook_checkpoints"

    print("\nNOTEBOOK CHECKPOINTS:")
    if backup.exists():
        for p in sorted(backup.glob("*")):
            print(p.name, "|", p.stat().st_size, "bytes")
    else:
        print("Pasta não encontrada:", backup)

    manifests = ROOT / "artifacts" / "manifests"
    print("\nMANIFESTS:")
    if manifests.exists():
        for p in sorted(manifests.glob("*")):
            print(p.name)
    else:
        print("Pasta não encontrada:", manifests)

print("=" * 65)

Mounted at /content/drive
AGRI-INTELLIGENCE — RECUPERAÇÃO DO CHECKPOINT
DRIVE ROOT: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence

CONTEÚDO PRINCIPAL:
[DIR ] artifacts
[DIR ] data
[DIR ] logs
[DIR ] state

NOTEBOOK CHECKPOINTS:
Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb | 313379 bytes

MANIFESTS:
l2_global_quota_fix_report.json
l2_median_split_fix.json
l3_final_candidate_selection.json
l3_optimized_selection.json
l3_pairing_overlap_audit.json
l3_selected_blocks.json
l3_spatial_preflight.json
l3_stratified_selection.json
l3_tile_gt_audit.json
phase_2_5_l2_engineering_review.json
phase_2_5_l2_implementation.json
phase_2_5_source_manifest.json


In [5]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

print("=" * 70)
print("AGRI-INTELLIGENCE — INVENTÁRIO DE RECUPERAÇÃO")
print("=" * 70)

for folder_name in ["state", "logs", "artifacts", "data"]:
    folder = ROOT / folder_name
    print(f"\n[{folder_name.upper()}]")

    if not folder.exists():
        print("Pasta ausente")
        continue

    files = sorted(
        folder.rglob("*"),
        key=lambda p: (len(p.parts), str(p).lower())
    )

    if not files:
        print("Vazia")
        continue

    for p in files:
        if p.is_file():
            print(f"FILE | {p.relative_to(ROOT)} | {p.stat().st_size:,} bytes")
        elif len(p.relative_to(ROOT).parts) <= 3:
            print(f"DIR  | {p.relative_to(ROOT)}/")

print("\n" + "=" * 70)
print("ARQUIVOS DE RECUPERAÇÃO ENCONTRADOS")
print("=" * 70)

patterns = ["*.bundle", "*.zip", "*.tar", "*.tar.gz", "*.patch", "*.json", "*.ipynb"]

for pattern in patterns:
    matches = list(ROOT.rglob(pattern))
    print(f"\n{pattern}: {len(matches)}")
    for p in matches[:30]:
        print(" -", p.relative_to(ROOT))

print("\n" + "=" * 70)

AGRI-INTELLIGENCE — INVENTÁRIO DE RECUPERAÇÃO

[STATE]
FILE | state/phase_2_5_bundle_recovery.json | 311 bytes
FILE | state/runtime_environment.json | 438 bytes

[LOGS]
Vazia

[ARTIFACTS]
DIR  | artifacts/backups/
DIR  | artifacts/manifests/
DIR  | artifacts/backups/l2_correction_20261110_022205/
DIR  | artifacts/backups/l2_global_quota_fix_20261110_022717/
DIR  | artifacts/backups/l2_median_fix_20261110_022402/
DIR  | artifacts/backups/notebook_checkpoints/
FILE | artifacts/manifests/l2_global_quota_fix_report.json | 694 bytes
FILE | artifacts/manifests/l2_median_split_fix.json | 1,042 bytes
FILE | artifacts/manifests/l3_final_candidate_selection.json | 81,608 bytes
FILE | artifacts/manifests/l3_optimized_selection.json | 100,034 bytes
FILE | artifacts/manifests/l3_pairing_overlap_audit.json | 3,675 bytes
FILE | artifacts/manifests/l3_selected_blocks.json | 54,662 bytes
FILE | artifacts/manifests/l3_spatial_preflight.json | 5,735 bytes
FILE | artifacts/manifests/l3_stratified_selectio

In [6]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

files = [
    ROOT / "state/phase_2_5_bundle_recovery.json",
    ROOT / "state/runtime_environment.json",
]

for path in files:
    print("\n" + "=" * 70)
    print(path.name)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2, ensure_ascii=False))

for folder in [
    ROOT / "data/interim/phase_2_5",
    ROOT / "data/processed/phase_2_5",
]:
    print("\n" + "=" * 70)
    print("DIRETÓRIO:", folder)
    print("=" * 70)

    if folder.exists():
        items = list(folder.rglob("*"))
        print("Itens:", len(items))
        for p in items[:30]:
            if p.is_file():
                print(p.relative_to(ROOT), "|", p.stat().st_size, "bytes")
            else:
                print(p.relative_to(ROOT), "/")


phase_2_5_bundle_recovery.json
{
  "bundle_path": "/content/drive/MyDrive/agri_phase25/phase25_protocol.bundle",
  "source_ref": "refs/heads/feat/phase-2-5-first-real-experiment",
  "imported_ref": "refs/remotes/backup/phase25",
  "commit": "0cb46c62191c06125c20f0636e66bdf4fa74bfbe",
  "verified_at_utc": "2026-11-10T01:45:27.657747+00:00"
}

runtime_environment.json
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "modules": {
    "numpy": true,
    "pandas": true,
    "rasterio": true,
    "geopandas": true,
    "shapely": true,
    "requests": true,
    "PIL": true
  },
  "branch": "feat/platform-notebook-v1",
  "commit": "d8611bda5765449a9b79fb88d824476172cf89f2",
  "checked_at_utc": "2026-11-10T01:45:27.696340+00:00"
}

DIRETÓRIO: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/interim/phase_2_5
Itens: 0

DIRETÓRIO: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/processed/phase_2_5
Iten

In [7]:

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/edu-moraess/agri-intelligence.git"
BRANCH = "feat/platform-notebook-v1"
DEST = Path("/content/agri-intelligence-platform-v1")

def run(cmd, cwd=None):
    p = subprocess.run(
        cmd, cwd=cwd, text=True,
        capture_output=True
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"FALHA: {' '.join(map(str, cmd))}\n{p.stderr}"
        )
    return p.stdout.strip()

print("RECUPERANDO REPOSITÓRIO...")

if DEST.exists():
    raise RuntimeError(
        f"O destino já existe: {DEST}. Não vou sobrescrever."
    )

run([
    "git", "clone",
    "--single-branch",
    "--branch", BRANCH,
    REPO_URL,
    str(DEST)
])

branch = run(["git", "branch", "--show-current"], DEST)
head = run(["git", "rev-parse", "HEAD"], DEST)
remote = run(["git", "remote", "get-url", "origin"], DEST)
status = run(["git", "status", "--short"], DEST)

print("\n" + "=" * 65)
print("RECUPERAÇÃO DO REPOSITÓRIO")
print("=" * 65)
print("REMOTE:", remote)
print("BRANCH:", branch)
print("HEAD:", head)
print("WORKTREE:", status or "CLEAN")
print("CHECKPOINT LOCAL ESPERADO: fc6452b9b177439825175d9ed4d8d7f7206c401a")
print("CHECKPOINT PRESENTE NO CLONE:",
      "SIM" if run([
          "git", "cat-file", "-e",
          "fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}"
      ], DEST) == "" else "NÃO")
print("=" * 65)

RECUPERANDO REPOSITÓRIO...

RECUPERAÇÃO DO REPOSITÓRIO
REMOTE: https://github.com/edu-moraess/agri-intelligence.git
BRANCH: feat/platform-notebook-v1
HEAD: d8611bda5765449a9b79fb88d824476172cf89f2
WORKTREE: CLEAN
CHECKPOINT LOCAL ESPERADO: fc6452b9b177439825175d9ed4d8d7f7206c401a


RuntimeError: FALHA: git cat-file -e fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}
fatal: Not a valid object name fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}


In [8]:

from pathlib import Path
import shutil
import subprocess
import json

REPO = Path("/content/agri-intelligence-platform-v1")
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

def git(args):
    p = subprocess.run(
        ["git", *args], cwd=REPO,
        capture_output=True, text=True
    )
    if p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

assert REPO.exists() and (REPO / ".git").exists()
assert git(["branch", "--show-current"]) == "feat/platform-notebook-v1"

print("=" * 65)
print("RECONSTRUÇÃO DO CHECKPOINT — PHASE 2.5")
print("=" * 65)

# 1. Restaurar manifests experimentais
src = DRIVE / "artifacts/manifests"
dst = REPO / "docs/experiments/phase_2_5/manifests"
dst.mkdir(parents=True, exist_ok=True)

manifest_count = 0
for f in src.glob("*.json"):
    shutil.copy2(f, dst / f.name)
    manifest_count += 1

# 2. Restaurar implementação L2 mais recente
backup = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"

assert (backup / "spatial_split.py").exists()
assert (backup / "test_spatial_split.py").exists()

code_dst = REPO / "src/agri_intelligence/data"
test_dst = REPO / "tests/unit"
code_dst.mkdir(parents=True, exist_ok=True)
test_dst.mkdir(parents=True, exist_ok=True)

shutil.copy2(backup / "spatial_split.py", code_dst / "spatial_split.py")
shutil.copy2(backup / "test_spatial_split.py", test_dst / "test_spatial_split.py")

# 3. Preservar notebook capturado da sessão anterior
notebook_src = DRIVE / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"
notebook_dst = REPO / "notebooks/Agri_Intelligence_Platform_v1.ipynb"

assert notebook_src.exists()
notebook_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(notebook_src, notebook_dst)

# 4. Preservar protocolo L2 disponível no backup
protocol_src = DRIVE / "artifacts/backups/l2_correction_20261110_022205/PHASE_2_5_PROTOCOL.md"
protocol_dst = REPO / "docs/experiments/phase_2_5/PHASE_2_5_PROTOCOL_L2_BACKUP.md"

if protocol_src.exists():
    protocol_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(protocol_src, protocol_dst)

# 5. Criar registro explícito de recuperação
recovery = {
    "status": "RECOVERED_FROM_DRIVE_ARTIFACTS",
    "previous_checkpoint_commit": "fc6452b9b177439825175d9ed4d8d7f7206c401a",
    "previous_checkpoint_commit_available": False,
    "recovered_base_commit": git(["rev-parse", "HEAD"]),
    "branch": git(["branch", "--show-current"]),
    "manifests_restored": manifest_count,
    "notebook_restored": str(notebook_dst),
    "l2_source_restored": str(code_dst / "spatial_split.py"),
    "l2_test_restored": str(test_dst / "test_spatial_split.py"),
    "data_source": str(DRIVE / "data/raw/zenodo_15235602"),
    "training_started": False
}

recovery_path = REPO / "docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json"
recovery_path.parent.mkdir(parents=True, exist_ok=True)
recovery_path.write_text(
    json.dumps(recovery, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nARQUIVOS RESTAURADOS")
print("Manifests:", manifest_count)
print("Notebook: OK")
print("L2 implementation: OK")
print("L2 test: OK")
print("Recovery record: OK")

print("\nGIT STATUS")
print(git(["status", "--short"]))

print("\nBASE:", git(["rev-parse", "--short", "HEAD"]))
print("BRANCH:", git(["branch", "--show-current"]))
print("\nRECONSTRUÇÃO DOS ARTEFATOS: CONCLUÍDA")
print("Nenhum treinamento ou push executado.")

RECONSTRUÇÃO DO CHECKPOINT — PHASE 2.5

ARQUIVOS RESTAURADOS
Manifests: 12
Notebook: OK
L2 implementation: OK
L2 test: OK
Recovery record: OK

GIT STATUS
M notebooks/Agri_Intelligence_Platform_v1.ipynb
?? docs/experiments/
?? src/agri_intelligence/data/spatial_split.py
?? tests/unit/

BASE: d8611bd
BRANCH: feat/platform-notebook-v1

RECONSTRUÇÃO DOS ARTEFATOS: CONCLUÍDA
Nenhum treinamento ou push executado.


In [9]:

from pathlib import Path
import subprocess

REPO = Path("/content/agri-intelligence-platform-v1")

required = [
    "src/agri_intelligence/data/spatial_split.py",
    "src/agri_intelligence/data/gt_conversion.py",
    "tests/unit/test_spatial_split.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py",
    "docs/PHASE_2_5_PROTOCOL.md",
    "notebooks/Agri_Intelligence_Platform_v1.ipynb",
    "docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json",
]

print("AGRI-INTELLIGENCE — RECOVERY GATE")
print("=" * 65)

for item in required:
    p = REPO / item
    print(("PASS" if p.is_file() else "MISSING"), item)

print("\nMANIFESTS:", len(list(
    (REPO / "docs/experiments/phase_2_5/manifests").glob("*.json")
)))

p = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO, capture_output=True, text=True
)
print("\nGIT STATUS:\n", p.stdout or "CLEAN")

print("=" * 65)

AGRI-INTELLIGENCE — RECOVERY GATE
PASS src/agri_intelligence/data/spatial_split.py
MISSING src/agri_intelligence/data/gt_conversion.py
PASS tests/unit/test_spatial_split.py
MISSING tests/unit/test_gt_conversion.py
MISSING tests/unit/test_l2_integration.py
MISSING docs/PHASE_2_5_PROTOCOL.md
PASS notebooks/Agri_Intelligence_Platform_v1.ipynb
PASS docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json

MANIFESTS: 12

GIT STATUS:
  M notebooks/Agri_Intelligence_Platform_v1.ipynb
?? docs/experiments/
?? src/agri_intelligence/data/spatial_split.py
?? tests/unit/



In [10]:

import subprocess
from pathlib import Path

REPO = Path("/content/agri-intelligence-platform-v1")

def git(args, check=True):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True
    )
    if check and p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

print("=" * 70)
print("BUSCA DE ARQUIVOS AUSENTES NO HISTÓRICO REMOTO")
print("=" * 70)

print("\n1. Atualizando referências remotas...")
print(git(["fetch", "origin", "--prune"]))

print("\n2. Branches remotas:")
print(git(["branch", "-r"]))

targets = [
    "src/agri_intelligence/data/gt_conversion.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py",
    "docs/PHASE_2_5_PROTOCOL.md",
]

print("\n3. Histórico dos arquivos:")
for path in targets:
    print(f"\n--- {path} ---")
    result = git(
        ["log", "--all", "--format=%H | %ad | %s",
         "--date=short", "--", path],
        check=False
    )
    print(result if result else "NÃO ENCONTRADO NO HISTÓRICO")

print("\n4. Verificando a branch de recuperação anterior:")
ref = "origin/feat/phase-2-5-first-real-experiment"
exists = subprocess.run(
    ["git", "rev-parse", "--verify", ref],
    cwd=REPO, capture_output=True, text=True
).returncode == 0

if exists:
    print("REF:", ref)
    print("HEAD:", git(["rev-parse", ref]))
    for path in targets:
        result = subprocess.run(
            ["git", "cat-file", "-e", f"{ref}:{path}"],
            cwd=REPO, capture_output=True
        )
        print(("FOUND " if result.returncode == 0 else "MISSING "), path)
else:
    print("Branch anterior não encontrada no remoto.")

print("\n" + "=" * 70)
print("BUSCA CONCLUÍDA — NENHUM ARQUIVO FOI ALTERADO")
print("=" * 70)

BUSCA DE ARQUIVOS AUSENTES NO HISTÓRICO REMOTO

1. Atualizando referências remotas...


2. Branches remotas:
origin/feat/platform-notebook-v1

3. Histórico dos arquivos:

--- src/agri_intelligence/data/gt_conversion.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- tests/unit/test_gt_conversion.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- tests/unit/test_l2_integration.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- docs/PHASE_2_5_PROTOCOL.md ---
NÃO ENCONTRADO NO HISTÓRICO

4. Verificando a branch de recuperação anterior:
Branch anterior não encontrada no remoto.

BUSCA CONCLUÍDA — NENHUM ARQUIVO FOI ALTERADO


In [11]:

import json
import re
import hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
NOTEBOOK = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

assert NOTEBOOK.exists(), f"Notebook ausente: {NOTEBOOK}"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "gt_conversion",
    "assign_spatial_splits",
    "spatial_split",
    "Polygon",
    "MultiPolygon",
    "ImageFrame",
    "TileRecord",
    "DatasetManifest",
    "PHASE_2_5_PROTOCOL",
    "PARTIAL_LT_50",
    "OUTSIDE_SELECTED_TILES",
    "audit_split_integrity",
]

matches = []

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    found = [term for term in terms if term.lower() in source.lower()]

    if found:
        matches.append({
            "cell": i,
            "terms": found,
            "source": source
        })

out = ROOT / "artifacts/backups/notebook_recovery_candidates"
out.mkdir(parents=True, exist_ok=True)

report = []

for item in matches:
    filename = f"cell_{item['cell']:02d}.py"
    path = out / filename
    path.write_text(item["source"], encoding="utf-8")

    report.append({
        "cell": item["cell"],
        "file": filename,
        "terms": item["terms"],
        "lines": len(item["source"].splitlines()),
        "sha256": hashlib.sha256(
            item["source"].encode()
        ).hexdigest()
    })

report_path = out / "recovery_index.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 65)
print("NOTEBOOK — RECUPERAÇÃO DE CÓDIGO")
print("=" * 65)
print("Total de células:", len(nb.get("cells", [])))
print("Células relevantes:", len(matches))

for item in report:
    print(
        f"\nCélula {item['cell']} | {item['lines']} linhas"
        f"\nTermos: {', '.join(item['terms'])}"
        f"\nArquivo: {item['file']}"
    )

print("\nÍNDICE:", report_path)
print("CANDIDATOS:", out)
print("=" * 65)

NOTEBOOK — RECUPERAÇÃO DE CÓDIGO
Total de células: 35
Células relevantes: 16

Célula 16 | 52 linhas
Termos: gt_conversion, spatial_split, PHASE_2_5_PROTOCOL
Arquivo: cell_16.py

Célula 17 | 111 linhas
Termos: PHASE_2_5_PROTOCOL
Arquivo: cell_17.py

Célula 18 | 732 linhas
Termos: gt_conversion, assign_spatial_splits, spatial_split, Polygon, MultiPolygon, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES, audit_split_integrity
Arquivo: cell_18.py

Célula 19 | 55 linhas
Termos: gt_conversion, spatial_split
Arquivo: cell_19.py

Célula 20 | 452 linhas
Termos: gt_conversion, spatial_split, Polygon, MultiPolygon, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES
Arquivo: cell_20.py

Célula 21 | 526 linhas
Termos: gt_conversion, spatial_split, Polygon, MultiPolygon, ImageFrame, TileRecord, DatasetManifest, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES
Arquivo: cell_21.py

Célula 22 | 29 linhas
Termos: spatial_split
Arquivo: cell_22.py

Célula 23 | 246 linhas
Termos: